# exp128_trajectory_local_typewell_self_gr_switch_audit train

Train-side audit for local switching between typewell GR observation cost and same-horizontal self-GR prefix matches.

## Contents

1. Setup and configuration
2. Input checks
3. Run local switch audit
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd
import yaml

from settings import ExperimentPaths, load_config
from trajectory_local_typewell_self_gr_switch_audit import run_audit

paths = ExperimentPaths()
config = load_config()
print('experiment:', config['experiment']['name'])
print('route:', config['experiment']['route'])
print('parent:', config['lineage']['parent'])
print('status:', config['experiment']['status'])
print('kaggle train dir:', paths.train_data_dir)
print('artifacts dir:', paths.artifacts_dir)
print('base candidates:', config['model']['base_candidates'])
print('local switch:', json.dumps(config['model']['local_switch'], indent=2))

## 2. Input checks

In [ ]:
expected_cache = config['data']['exp099_train_feature_cache_local']
expected_schema = config['data']['exp099_train_feature_schema_local']
print('configured exp099 cache:', expected_cache)
print('configured exp099 schema:', expected_schema)
print('train data exists:', paths.train_data_dir.exists())
if paths.train_data_dir.exists():
    horizontal_files = sorted(paths.train_data_dir.glob('*__horizontal_well.csv'))
    typewell_files = sorted(paths.train_data_dir.glob('*__typewell.csv'))
    print('horizontal files:', len(horizontal_files))
    print('typewell files:', len(typewell_files))
    if horizontal_files:
        preview = pd.read_csv(horizontal_files[0], nrows=5)
        display(preview)
else:
    print('Kaggle input will be resolved at runtime.')

## 3. Run local switch audit

In [ ]:
summary = run_audit(config=config, paths=paths)
print(json.dumps({
    'rows': summary['rows'],
    'wells': summary['wells'],
    'best_candidate': summary['best_candidate'],
    'likpf_baseline': summary['likpf_baseline'],
    'delta_best_minus_likpf_rmse': summary['delta_best_minus_likpf_rmse'],
}, indent=2))

## 4. Metrics and artifacts

In [ ]:
artifact_paths = summary['artifacts']
for name, value in artifact_paths.items():
    print(name, value)

candidate_metrics = pd.read_csv(artifact_paths['candidate_metrics'])
bucket_metrics = pd.read_csv(artifact_paths['bucket_metrics'])
by_well = pd.read_csv(artifact_paths['by_well'])
signal_metrics = pd.read_csv(artifact_paths['signal_metrics'])
window_diagnostics = pd.read_csv(artifact_paths['window_diagnostics'])

display(candidate_metrics.head(20))
display(signal_metrics)
display(bucket_metrics.head(30))
display(by_well.sort_values('delta_vs_likpf_rmse', ascending=False).head(20))
display(window_diagnostics.head(20))

with open(paths.metrics_path) as fp:
    metrics_json = json.load(fp)
print(json.dumps(metrics_json, indent=2))